In [46]:
import torch
import pandas as pd
from read_jsonl import read_jsonl
from transformers import DistilBertTokenizer,DistilBertForSequenceClassification

In [2]:
df = read_jsonl("DB-bio/train_sft.jsonl")
df.rename(columns={"output": "anonymized", "input": "original"}, inplace=True)
df = df.drop(columns=["people", "label"])
df.columns

Index(['anonymized', 'original'], dtype='object')

In [10]:
tokenizer_distilbert = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model_distil = DistilBertForSequenceClassification.from_pretrained("./results/distilbert/checkpoint-170")
model_distil.eval()
embedding_matrix = model_distil.get_input_embeddings().weight
embedding_matrix.shape

torch.Size([30522, 768])

In [62]:
def build_anonymization_dict(row, tok, embeddings):
    ids_original = tok(row["original"], truncation=True, max_length=512)["input_ids"]
    ids_anonymized = tok(row["anonymized"], truncation=True, max_length=512)["input_ids"]
    embeddings_original= embeddings[ids_original]
    embeddings_anonymized = embeddings[ids_anonymized]
    with torch.no_grad():
        sim = embeddings_anonymized @ embeddings_original.T
    ids_original_match = torch.argmax(sim, dim=0)
    ids_anonymized_match = torch.argmax(sim, dim=1)
    ids_original_replacement = {token: ids_anonymized[ids_original_match[i]] for i,token in enumerate(ids_original) 
                                if ids_anonymized[ids_original_match[i]] != token}
    ids_anonymized_replacement = {token: ids_original[ids_anonymized_match[i]] for i,token in enumerate(ids_anonymized) 
                                  if ids_original[ids_anonymized_match[i]] != token}
    return ids_original_replacement, ids_anonymized_replacement

In [67]:
df_test = df.head(1).copy()
df_test['anonymized'].tolist()

['A person (born in the early 20th century – passed away in the late 20th century) was a British chess player and writer. This person contributed articles to the British Chess Magazine (BCM) from the early 1930s to the early 1980s, and to the British Chess Federation\'s publications Newsflash and Chess Moves from the mid-1970s to the early 1990s. A notable chess figure called this person "one of the best writers on chess that I know". In a notable chess book, this figure reproduced in toto this person\'s account, first published in the BCM during the mid-1940s, of a significant 19th-century chess match. After this person told the notable chess figure of a game they had lost quickly (1.e4 e5 2.Nc3 Nf6 3.Bc4 Nxe4 4.Bxf7+?! Kxf7 5.Nxe4 Nc6 6.Qf3+ Kg8?? 7.Ng5! 1-0, a match in a London league in the late 1940s), the figure affectionately christened this person "the Badmaster", a facetious counterpoint to the more familiar title Grandmaster. This person later adopted the sobriquet as a pseud

In [ ]:

df_test[["original_dict", "anonymized_dict"]] = df_test.apply(build_anonymization_dict, args=[tokenizer_distilbert, embedding_matrix], axis=1).apply(pd.Series)

In [64]:
for k,v in df_test["original_dict"].tolist()[0].items():
    token = tokenizer_distilbert.decode([k])
    token_replacement = tokenizer_distilbert.decode([v])
    print("{} --> {}".format(token, token_replacement))

geoffrey --> ##quet
ha --> ##quet
##rber --> ##quet
dig --> sob
##gle --> ##quet
december --> unaffected
1902 --> ##quet
13 --> sob
february --> unaffected
1993 --> unaffected
1933 --> unaffected
1981 --> unaffected
1974 --> unaffected
1992 --> unaffected
c --> ##quet
h --> ##quet
o --> ##rew
d --> ##quet
alexander --> ##rew
\ --> ##tious
november --> unaffected
1943 --> ##quet
facto --> ##cratic
1843 --> ##quet
world --> sob
championship --> ##quet
st --> sob
##au --> ##quet
##nton --> ##rew
ama --> ##mini
##nt --> ##quet
seven --> sob
david --> ##rew
banks --> ##rew
1949 --> ##quet
1986 --> unaffected
edward --> ##rew
winter --> ##quet
lincoln --> ##quet
bottom --> ##rew
1922 --> ##quet
g --> sob
tar --> sob
##ta --> sob
##kow --> ##rew
##er --> ##rew


torch.Size([512, 768])


In [36]:
df_test

,anonymized,original,sim_dict
0,A person (born in the early 20th century – pas...,Geoffrey Harber Diggle (6 December 1902 – 13 F...,"({101: 101, 11023: 12647, 5292: 12647, 20473: ..."


In [30]:
ten = torch.tensor([[1,2],[4,3]]) 
ten

tensor([[1, 2],
        [4, 3]])

In [29]:
torch.argmax(ten, dim=0), torch.argmax(ten, dim=1)

(tensor([1, 1]), tensor([1, 0]))